In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
from sklearn.datasets import load_digits
import pandas as pd

digits = load_digits()

X = pd.DataFrame(digits.data)
y = pd.Series(digits.target, name="digit")
y.head(100)

0     0
1     1
2     2
3     3
4     4
     ..
95    6
96    8
97    4
98    3
99    1
Name: digit, Length: 100, dtype: int64

# Creating The Model

In [3]:
def minkowski_distance(row, sample, p):
    # I chose mikowski distance because it is flexible (we can choose if p=1 then we'll have euclidian distance and if p=2 then we'll have manhatten distance)
    distance = np.sum(np.abs(row - sample) ** p) ** (1/p)
    return distance


In [4]:
def get_neighbors(X, y, sample, p):
    # instead of sorting all the data , we only need distance and the y corresponding to all the features but not the features its self
    neighbors = []
    for index, row in X.iterrows():
        distance = minkowski_distance(row, sample, p)
        corresponding_y = y.iloc[index]
        neighbors.append((distance, corresponding_y))
    neighbors.sort()
    return neighbors

In [5]:
def weighted_vote(neighbors):
    votes = {}
    for distance, label in neighbors:
        if distance == 0:
            return label
        weight = 1 / distance
        if label not in votes:
            votes[label] = weight
        else:
            votes[label] += weight
    prediction = max(votes, key=votes.get)
    return prediction
    

In [6]:
def predict(X_train, y_train, sample, k, p):
    neighbors = get_neighbors(X_train, y_train, sample, p)
    neighbors = neighbors[:k]
    prediction = weighted_vote(neighbors)
    return prediction


In [11]:
def predict_multiple_lines(X_train, y_train, X_test, k, p):
    predictions = []
    for _, row in X_test.iterrows():
        predictions.append(predict(X_train, y_train, row, k, p))
    return predictions

In [12]:
def accuracy(y_true, y_pred):
    correct = 0

    for true, pred in zip(y_true, y_pred):
        if true == pred:
            correct += 1

    return correct / len(y_true)

# Predicting result

In [14]:
# splitting the dataset
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)
# resetting the index
X_train = X_train.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)


prediction_me = predict_multiple_lines(
    X_train,
    y_train,
    X_test,
    k=5,
    p=2
)

print(accuracy(y_test, prediction_me))

0.9861111111111112


# Creating the same model using sklearn

In [15]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
model = KNeighborsClassifier(n_neighbors=5,weights="distance",metric="minkowski",p=2)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)


# my model vs scikit-learn 

In [16]:
print(f" my accuracy :{accuracy(prediction_me, y_test)} vs scikit-learn's :{accuracy_score(y_test, y_pred)}")

 my accuracy :0.9861111111111112 vs scikit-learn's :0.9861111111111112


# Conclusion : it gives the same accuracy